In [ ]:
#ok for svm and knn
!pip install numpy pandas scikit-learn qiskit qiskit-machine-learning

In [ ]:
# Run this cell FIRST — restart kernel after it finishes
!pip install qiskit-ibm-runtime qiskit-machine-learning qiskit-algorithms --quiet

In [ ]:
import numpy as np
import pandas as pd
df=pd.read_csv('/content/survey lung cancer.csv')

In [ ]:
dfen2 = df.copy()
dfen2['GENDER'] = dfen2['GENDER'].map({'F':0,'M':1})
dfen2['LUNG_CANCER'] = dfen2['LUNG_CANCER'].map({'NO':0,'YES':1})

In [ ]:
# ------------------------------------------
#  Install imblearn once (if not installed)
# ------------------------------------------
!pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# ------------------------------------------
# 1️⃣  Verify dataset & identify target column
# ------------------------------------------
# You already have dfs
print("Original shape:", dfen2.shape)
print("Original class distribution:")
print(dfen2['LUNG_CANCER'].value_counts())

# ------------------------------------------
# 2️⃣  Separate features (X) and target (y)
# ------------------------------------------
X = dfen2.drop(columns=['LUNG_CANCER']).values
y = dfen2['LUNG_CANCER'].values

# ------------------------------------------
# 3️⃣  Apply SMOTE only on the minority class
# ------------------------------------------
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# ------------------------------------------
# 4️⃣  Rebuild into a balanced DataFrame
# ------------------------------------------
columns = dfen2.drop(columns=['LUNG_CANCER']).columns
dfs_smotes = pd.DataFrame(X_res, columns=columns)
dfs_smotes['LUNG_CANCER'] = y_res


# ------------------------------------------
# 5️⃣  Check the new class balance
# ------------------------------------------
print("\nAfter SMOTE:")
print(dfs_smotes['LUNG_CANCER'].value_counts())
print("New shape:", dfs_smotes.shape)

In [ ]:
dff=dfs_smotes[:]
dff

In [ ]:
dff.info()

**<h1>Qsvm**

In [ ]:
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)
from sklearn.svm import SVC

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityQuantumKernel

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
N_RUNS = 1
TEST_SIZE = 0.8
NUM_REPEATS = 1       # repeated random subsampling
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "qsvm_fixed_features_results.csv"
TARGET = "LUNG_CANCER"

# Fixed feature sequence (0-based indexing)
FIXED_FEATURES_IDX = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

# ============================================================
# LOAD DATA (replace dfs_smotes with your actual dataframe)
# ============================================================
try:
    dfs_smotes
except NameError:
    print("Creating synthetic dataset for demonstration...")
    np.random.seed(42)
    n_samples = 200
    n_features = 15
    X_pos = np.random.randn(n_samples // 2, n_features) + 1
    X_neg = np.random.randn(n_samples // 2, n_features) - 1
    X = np.vstack([X_pos, X_neg])
    y = np.array([1] * (n_samples // 2) + [0] * (n_samples // 2))
    columns = [f"feature_{i}" for i in range(n_features)]
    dfs_smotes = pd.DataFrame(X, columns=columns)
    dfs_smotes[TARGET] = y
    print(f"Sample dataset created: {dfs_smotes.shape}")

X_raw = dfs_smotes.iloc[:, FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)
feature_names = dfs_smotes.columns[FIXED_FEATURES_IDX].tolist()
print(f"Using fixed features: {feature_names}")

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

# ============================================================
# QISKIT CUSTOM FEATURE MAP (Dense Angle Encoding)
# ============================================================
def create_dense_angle_feature_map(num_features, circuit_depth=1, n_reuploads=1):
    """
    Creates a parameterized Qiskit QuantumCircuit implementing
    the dense angle encoding logic requested.
    """
    # Create abstract parameters for the data features
    x = ParameterVector('x', num_features)
    qc = QuantumCircuit(num_features)

    for _ in range(circuit_depth):
        for _ in range(n_reuploads):
            # Rotations
            for i in range(num_features):
                qc.rx(x[i], i)
                qc.ry(2.0 * x[i], i)
                qc.rz(0.5 * x[i], i)

            # Linear Entanglement (CNOTs)
            for i in range(num_features - 1):
                qc.cx(i, i + 1)

    return qc

# ============================================================
# QSVM PREDICTION FUNCTION (Using Qiskit)
# ============================================================
def predict_qiskit_svc(X_train, y_train, X_test):
    n_features = X_train.shape[1]

    # Generate the custom Qiskit Dense Angle Feature Map
    feature_map = create_dense_angle_feature_map(num_features=n_features, circuit_depth=1, n_reuploads=1)

    # Initialize FidelityQuantumKernel with our custom feature map
    qkernel = FidelityQuantumKernel(feature_map=feature_map)

    # Compute kernel matrices internally (highly optimized by Qiskit)
    kernel_train = qkernel.evaluate(x_vec=X_train)
    kernel_test = qkernel.evaluate(x_vec=X_test, y_vec=X_train)

    # Classical SVM using precomputed quantum kernel
    clf = SVC(kernel='precomputed', probability=True)
    clf.fit(kernel_train, y_train)

    # Use decision_function and map to probability via sigmoid approximation
    probs = clf.decision_function(kernel_test)
    probs = 1 / (1 + np.exp(-probs))  # sigmoid mapping
    return probs

# ============================================================
# MAIN LOOP WITH REPEATED RANDOM SUBSAMPLING
# ============================================================
summary = []
total_jobs = NUM_REPEATS
completed_jobs = 0

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1, NUM_REPEATS + 1):
    # Train-Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED_BASE * split
    )

    # Imputation + Scaling
    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    # ---------- QSVM / SVM ----------
    start = time.time()
    y_prob = predict_qiskit_svc(X_train, y_train, X_test)
    runtime = time.time() - start

    # ---------- Metrics ----------
    metrics = compute_metrics(y_test, y_prob)
    completed_jobs += 1
    progress = (completed_jobs / total_jobs) * 100

    print(
        f"[{progress:6.2f}%] "
        f"QSVM_Fixed | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row = {
        "Split": split,
        "Accuracy": metrics["Accuracy"],
        "ROC_AUC": metrics["ROC-AUC"],
        "F1": metrics["F1"],
        "Precision": metrics["Precision"],
        "Sensitivity": metrics["Sensitivity"],
        "Specificity": metrics["Specificity"],
        "Kappa": metrics["Kappa"],
        "Runtime_sec": runtime
    }
    summary.append(row)
    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

# ============================================================
# FINAL SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)

print("\n===== FINAL SORTED RESULTS =====")
print(summary_df)
print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")

**<h1>Qknn**

In [ ]:
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityQuantumKernel

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
N_RUNS = 1
TEST_SIZE = 0.8
NUM_REPEATS = 65
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "qknn_fixed_features_results.csv"
TARGET = "LUNG_CANCER"

# QKNN parameter
K_NEIGHBORS = 3

# Fixed feature sequence (0-based indexing)
FIXED_FEATURES_IDX = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

# ============================================================
# LOAD DATA
# ============================================================
try:
    dfs_smotes
except NameError:
    print("Creating synthetic dataset for demonstration...")
    np.random.seed(42)
    n_samples = 200
    n_features = 15
    X_pos = np.random.randn(n_samples // 2, n_features) + 1
    X_neg = np.random.randn(n_samples // 2, n_features) - 1
    X = np.vstack([X_pos, X_neg])
    y = np.array([1] * (n_samples // 2) + [0] * (n_samples // 2))
    columns = [f"feature_{i}" for i in range(n_features)]
    dfs_smotes = pd.DataFrame(X, columns=columns)
    dfs_smotes[TARGET] = y
    print(f"Sample dataset created: {dfs_smotes.shape}")

X_raw = dfs_smotes.iloc[:, FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)
feature_names = dfs_smotes.columns[FIXED_FEATURES_IDX].tolist()
print(f"Using fixed features: {feature_names}")

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

# ============================================================
# QISKIT CUSTOM FEATURE MAP
# ============================================================
def create_dense_angle_feature_map(num_features, circuit_depth=1, n_reuploads=1):

    x = ParameterVector('x', num_features)
    qc = QuantumCircuit(num_features)

    for _ in range(circuit_depth):
        for _ in range(n_reuploads):

            for i in range(num_features):
                qc.rx(x[i], i)
                qc.ry(2.0 * x[i], i)
                qc.rz(0.5 * x[i], i)

            for i in range(num_features - 1):
                qc.cx(i, i + 1)

    return qc

# ============================================================
# QKNN PREDICTION FUNCTION (Using Qiskit)
# ============================================================
def predict_qiskit_qknn(X_train, y_train, X_test):

    n_features = X_train.shape[1]

    feature_map = create_dense_angle_feature_map(
        num_features=n_features,
        circuit_depth=1,
        n_reuploads=1
    )

    qkernel = FidelityQuantumKernel(feature_map=feature_map)

    kernel_train = qkernel.evaluate(x_vec=X_train)
    kernel_test = qkernel.evaluate(x_vec=X_test, y_vec=X_train)

    probs = []

    for i in range(kernel_test.shape[0]):

        sims = kernel_test[i]

        idx = np.argsort(sims)[-K_NEIGHBORS:]

        labels = y_train[idx]

        prob = np.mean(labels)

        probs.append(prob)

    return np.array(probs)

# ============================================================
# MAIN LOOP
# ============================================================
summary = []
total_jobs = NUM_REPEATS
completed_jobs = 0

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1, NUM_REPEATS + 1):

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE * split
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()

    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    start = time.time()

    y_prob = predict_qiskit_qknn(X_train, y_train, X_test)

    runtime = time.time() - start

    metrics = compute_metrics(y_test, y_prob)

    completed_jobs += 1
    progress = (completed_jobs / total_jobs) * 100

    print(
        f"[{progress:6.2f}%] "
        f"QKNN_Fixed | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row = {
        "Split": split,
        "Accuracy": metrics["Accuracy"],
        "ROC_AUC": metrics["ROC-AUC"],
        "F1": metrics["F1"],
        "Precision": metrics["Precision"],
        "Sensitivity": metrics["Sensitivity"],
        "Specificity": metrics["Specificity"],
        "Kappa": metrics["Kappa"],
        "Runtime_sec": runtime
    }

    summary.append(row)

    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

# ============================================================
# FINAL SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)

print("\n===== FINAL SORTED RESULTS =====")
print(summary_df)
print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")

**<h1>Qboost**

In [ ]:
# ============================================================
# CELL 1: RUN THIS ALONE → Then click Runtime → Restart runtime
# ============================================================
!pip uninstall -y qiskit qiskit-terra qiskit-aer qiskit-machine-learning qiskit-algorithms 2>/dev/null
!pip install qiskit==1.0.2
print("\n✅ DONE. Now click: Runtime → Restart runtime → Then run CELL 2")

In [ ]:
# ============================================================
# RUN AFTER RESTARTING RUNTIME (Requires Qiskit 1.0)
# ============================================================
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
TEST_SIZE = 0.8
NUM_REPEATS = 65
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "qboost_fixed_features_results.csv"
TARGET = "LUNG_CANCER"

FIXED_FEATURES_IDX = [8,10,9,13,11,14,5,3,6,2]
N_QUBITS = len(FIXED_FEATURES_IDX)

# ============================================================
# LOAD DATA
# ============================================================
try:
    dfs_smotes
    print(f"Using dataset: {dfs_smotes.shape}")
except NameError:
    raise ValueError("Dataset dfs_smotes not loaded")

X_raw = dfs_smotes.iloc[:, FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, thr=0.5):

    y_pred = (y_prob >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Sensitivity": tp/(tp+fn) if(tp+fn)>0 else 0,
        "Specificity": tn/(tn+fp) if(tn+fp)>0 else 0,
        "Kappa": cohen_kappa_score(y_true,y_pred)
    }

# ============================================================
# PARAMETERIZED QUANTUM CIRCUIT
# ============================================================
def create_parameterized_circuit(num_features, num_layers=2):

    qc = QuantumCircuit(num_features)

    x = ParameterVector('x', num_features)
    theta = ParameterVector('t', num_features * num_layers)

    # feature encoding
    for i in range(num_features):

        qc.rx(x[i], i)
        qc.ry(2*x[i], i)
        qc.rz(0.5*x[i], i)

    for i in range(num_features-1):
        qc.cx(i,i+1)

    # trainable ansatz
    idx = 0
    for _ in range(num_layers):

        for i in range(num_features):

            qc.ry(theta[idx], i)
            idx+=1

        for i in range(num_features-1):
            qc.cx(i,i+1)

    return qc, x, theta


# ============================================================
# QUANTUM WEAK LEARNER
# ============================================================
class QiskitWeakLearner:

    def __init__(self, num_features, num_layers=2, max_iter=40):

        self.num_features = num_features
        self.num_layers = num_layers
        self.max_iter = max_iter

        self.weights = np.random.uniform(-np.pi,np.pi,num_features*num_layers)

        self.qc,self.x_params,self.theta_params = create_parameterized_circuit(
            num_features,num_layers
        )

    def get_scores(self,X,weights):

        scores=[]

        for x_val in X:

            param_dict={self.x_params[i]:x_val[i] for i in range(self.num_features)}

            param_dict.update({
                self.theta_params[i]:weights[i]
                for i in range(len(weights))
            })

            bound_qc=self.qc.assign_parameters(param_dict)

            state=Statevector(bound_qc)

            probs=np.abs(state.data)**2

            score=0

            for i,p in enumerate(probs):

                bit=(i>>(self.num_features-1)) & 1
                z=1 if bit==0 else -1
                score+=z*p

            scores.append(score)

        return np.array(scores)

    def fit(self,X,y,sample_weights):

        y_trans=2*y-1

        def loss(weights):

            scores=self.get_scores(X,weights)

            return np.sum(sample_weights*(scores-y_trans)**2)

        res=minimize(
            loss,
            self.weights,
            method="COBYLA",
            options={"maxiter":self.max_iter}
        )

        self.weights=res.x

        return self

    def predict(self,X):

        scores=self.get_scores(X,self.weights)

        return (scores>0).astype(int)


# ============================================================
# QBOOST ENSEMBLE
# ============================================================
class QBoost:

    def __init__(self,n_estimators=3,num_layers=2,max_iter=40):

        self.n_estimators=n_estimators
        self.num_layers=num_layers
        self.max_iter=max_iter

        self.learners=[]
        self.alphas=[]

    def fit(self,X,y):

        n_samples=X.shape[0]

        sample_weights=np.ones(n_samples)/n_samples

        for _ in range(self.n_estimators):

            learner=QiskitWeakLearner(
                X.shape[1],
                self.num_layers,
                self.max_iter
            )

            learner.fit(X,y,sample_weights)

            y_pred=learner.predict(X)

            err=np.sum(sample_weights*(y_pred!=y))/np.sum(sample_weights)

            if err>=0.5:
                continue

            alpha=0.5*np.log((1-err)/max(err,1e-10))

            sample_weights*=np.exp(-alpha*(2*y-1)*(2*y_pred-1))
            sample_weights/=np.sum(sample_weights)

            self.learners.append(learner)
            self.alphas.append(alpha)

        return self

    def predict_proba(self,X):

        final_score=np.zeros(X.shape[0])

        for learner,alpha in zip(self.learners,self.alphas):

            score=learner.get_scores(X,learner.weights)

            final_score+=alpha*score

        prob=1/(1+np.exp(-final_score))

        return np.vstack([1-prob,prob]).T


# ============================================================
# MAIN LOOP
# ============================================================
summary=[]
total_jobs=NUM_REPEATS
completed_jobs=0

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

print("\nStarting QBoost experiment...\n")

for split in range(1,NUM_REPEATS+1):

    X_train,X_test,y_train,y_test=train_test_split(
        X_raw,y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE*split
    )

    imp=SimpleImputer(strategy="median")
    sc=StandardScaler()

    X_train=sc.fit_transform(imp.fit_transform(X_train))
    X_test=sc.transform(imp.transform(X_test))

    start=time.time()

    clf=QBoost(n_estimators=3,num_layers=2,max_iter=40)

    clf.fit(X_train,y_train)

    y_prob=clf.predict_proba(X_test)[:,1]

    runtime=time.time()-start

    metrics=compute_metrics(y_test,y_prob)

    completed_jobs+=1
    progress=(completed_jobs/total_jobs)*100

    print(
        f"[{progress:6.2f}%] "
        f"QBoost | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row={
        "Split":split,
        "Accuracy":metrics["Accuracy"],
        "ROC_AUC":metrics["ROC-AUC"],
        "F1":metrics["F1"],
        "Precision":metrics["Precision"],
        "Sensitivity":metrics["Sensitivity"],
        "Specificity":metrics["Specificity"],
        "Kappa":metrics["Kappa"],
        "Runtime_sec":runtime
    }

    summary.append(row)

    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH,index=False)


# ============================================================
# FINAL RESULTS
# ============================================================
summary_df=pd.DataFrame(summary)

summary_df=summary_df.sort_values(
    "Accuracy",
    ascending=False
).reset_index(drop=True)

print("\n===== FINAL SORTED RESULTS =====")

print(summary_df)

print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")

print("\nDONE.")